In [ ]:
import uproot
import pandas as pd
import awkward as ak
import matplotlib.pyplot as plt

tables = [
    "O2upgradecascades",
    "O2upgradev0s"
]

def load_tables(filename, tables):
    data_dict = {}

    with uproot.open(filename) as f:
        keys = f.keys()
        print(f"Keys in {filename}: {keys}")

        for table in tables:
            matched_keys = [
                key for key in keys
                if len(key.split("/")) == 2
                and key.split("/")[1].split(";")[0] == table
            ]
            # matched_keys = [key for key in keys if table == key.split("/")[1].replace(";1", "") and len(key.split("/")) == 2]
            # matched_keys = [key for key in keys if table in key]

            if not matched_keys:
                print(f"No keys found for table: {table}")
                continue

            table_dfs = []

            for key in matched_keys:
                array = f[key].arrays(library="ak")

                if len(array) == 0:
                    print(f"Empty array for key: {key}")
                    continue

                data = {}
                for k in array.fields:
                    v = array[k]
                    
                    if isinstance(ak.type(v).content, ak.types.NumpyType):
                        # scalar branch
                        data[k] = ak.to_numpy(v)
                    else:
                        # vector branch
                        data[k] = ak.to_list(v)

                df = pd.DataFrame(data)

                # df = pd.DataFrame({
                #     k: ak.to_list(array[k]) if ak.ndim(array[k]) > 1 else ak.to_numpy(array[k])
                #     for k in array.fields
                # })
                # df = pd.DataFrame({k: ak.to_list(v) for k, v in array.items()})
                # df = ak.to_dataframe(array)

                if df.empty:
                    print(f"Empty DataFrame for key: {key}")
                    continue

                # optional but VERY useful: avoid duplicate column collisions
                df = df.add_prefix(f"{table}__")

                table_dfs.append(df)

            if table_dfs:
                data_dict[table] = pd.concat(table_dfs, ignore_index=True)
            else:
                print(f"No usable data for table: {table}")

    return data_dict


# Load datasets
df_before_mod = load_tables("/home/mdicosta/LocalTestsO2/OnTheFlyTracker/AfterModifications/AO2DReco.root", tables)
df_after_mod  = load_tables("/home/mdicosta/LocalTestsO2/OnTheFlyTracker/AfterModifications/Tree.root", tables)

Keys in /home/mdicosta/LocalTestsO2/ReducedQvectors/BeforeModifications/Tree_2nd_3rd_harm.root: ['DF_2337256861756704;1', 'DF_2337256861756704/O2qvectordevs;1', 'DF_2337256861756704/O2qvectorsft0a;1', 'DF_2337256861756704/O2qvectorsft0avec;1', 'DF_2337256861756704/O2qvectorsft0c;1', 'DF_2337256861756704/O2qvectorsft0cvec;1', 'DF_2337256861756704/O2qvectorsft0m;1', 'DF_2337256861756704/O2qvectorsft0mvec;1', 'DF_2337256861756704/O2qvectorsfv0a;1', 'DF_2337256861756704/O2qvectorsfv0avec;1', 'DF_2337256861756704/O2qvectorstpcall;1', 'DF_2337256861756704/O2qvectorstpcavec;1', 'DF_2337256861756704/O2qvectorstpcneg;1', 'DF_2337256861756704/O2qvectorstpcnvec;1', 'DF_2337256861756704/O2qvectorstpcpos;1', 'DF_2337256861756704/O2qvectorstpcpvec;1', 'parentFiles;1']
Empty array for key: DF_2337256861756704/O2qvectorstpcpos;1
No usable data for table: O2qvectorstpcpos
Empty array for key: DF_2337256861756704/O2qvectorstpcneg;1
No usable data for table: O2qvectorstpcneg
Empty array for key: DF_23372

In [ ]:
print("Data loaded successfully.")
print(f"keys df_before_mod: {df_before_mod.keys()}")
print(f"keys df_after_mod: {df_after_mod.keys()}")

Data loaded successfully.
keys df_before_mod: dict_keys(['O2qvectordevs', 'O2qvectorsft0c', 'O2qvectorsft0a', 'O2qvectorsft0m', 'O2qvectorsfv0a', 'O2qvectorsft0cvec', 'O2qvectorsft0avec', 'O2qvectorsft0mvec', 'O2qvectorsfv0avec', 'O2qvectorstpcpvec', 'O2qvectorstpcnvec', 'O2qvectorstpcavec'])
keys df_after_mod: dict_keys(['O2qvectordevs', 'O2qvectorsft0c', 'O2qvectorsft0a', 'O2qvectorsft0m', 'O2qvectorsfv0a', 'O2qvectorstpcpos', 'O2qvectorstpcneg', 'O2qvectorstpcall', 'O2qvectorsft0cvec', 'O2qvectorsft0avec', 'O2qvectorsft0mvec', 'O2qvectorsfv0avec', 'O2qvectorstpcpvec', 'O2qvectorstpcnvec', 'O2qvectorstpcavec'])


In [ ]:
df_after_mod["O2upgradecascades"][121:133]

,O2qvectordevs__fCent,O2qvectordevs__fIsCalibrated,O2qvectordevs__fQvecRe_size,O2qvectordevs__fQvecRe,O2qvectordevs__fQvecIm_size,O2qvectordevs__fQvecIm,O2qvectordevs__fQvecAmp_size,O2qvectordevs__fQvecAmp
121,65.5,True,56,"[-0.07625675946474075, -0.036268312484025955, ...",56,"[-0.007404195610433817, -0.005147761665284634,...",14,"[2077.838134765625, 6876.03173828125, 8953.869..."
122,2.5,True,56,"[0.018890216946601868, 0.06135488674044609, 0....",56,"[0.025141337886452675, 0.029432371258735657, 0...",14,"[44080.171875, 149379.28125, 193459.390625, 13..."
123,99.5,True,56,"[0.8913489580154419, 0.925603985786438, 0.9366...",56,"[0.453317791223526, 0.45509830117225647, 0.476...",14,"[7.469818115234375, 42.37309265136719, 49.8429..."
124,32.5,True,56,"[-0.05043826997280121, -0.008089404553174973, ...",56,"[0.08706767857074738, 0.09157517552375793, 0.0...",14,"[13256.0234375, 44030.34375, 57286.37109375, 3..."
125,61.5,True,56,"[-0.014408975839614868, 0.026010774075984955, ...",56,"[0.058190733194351196, 0.06086094677448273, 0....",14,"[2790.222900390625, 10044.89453125, 12835.1142..."
126,99.5,True,56,"[-0.909275472164154, -0.875020444393158, -0.86...",56,"[0.4161948263645172, 0.4179753363132477, 0.398...",14,"[7.370378017425537, 9.20644474029541, 16.57682..."
127,83.5,True,56,"[0.09686485677957535, 0.1344052106142044, 0.13...",56,"[-0.03778679668903351, -0.03635744750499725, -...",14,"[480.7339782714844, 337.6964111328125, 818.430..."
128,79.5,True,56,"[-0.0017403363017365336, 0.03642777353525162, ...",56,"[0.2042085975408554, 0.2060411274433136, 0.206...",14,"[740.230224609375, 3213.04248046875, 3953.2729..."
129,99.5,True,56,"[-0.12093649804592133, -0.08668149262666702, -...",56,"[-0.5307364463806152, -0.5289559364318848, -0....",14,"[43.050777435302734, 2267.773193359375, 2310.8..."
130,70.5,True,56,"[0.1759616583585739, 0.21503300964832306, 0.21...",56,"[-0.07469581812620163, -0.07223356515169144, -...",14,"[1533.7265625, 7146.26953125, 8679.99609375, 5..."


In [ ]:
df_before_mod["O2upgradecascades"][121:133]

In [ ]:
df_after_mod["O2upgradev0s"][121:133]

,O2qvectorsft0m__fIsCalibrated,O2qvectorsft0m__fQvecFT0MRe,O2qvectorsft0m__fQvecFT0MIm,O2qvectorsft0m__fSumAmplFT0M
121,True,-0.341987,0.657673,8953.869141
122,True,1.547365,0.428462,193459.390625
123,True,-0.671115,-0.580509,49.842911
124,True,0.426441,0.890671,57286.371094
125,True,1.399353,0.268564,12835.114258
126,True,-1.470713,-0.572693,16.576822
127,True,-0.574801,-0.886148,818.430542
128,True,-0.564174,-0.051474,3953.272949
129,True,0.040469,-0.544981,2310.823975
130,True,0.353821,0.171130,8679.996094


In [ ]:
df_before_mod["O2upgradev0s"][121:133]

,O2qvectorsft0m__fIsCalibrated,O2qvectorsft0m__fQvecFT0MRe,O2qvectorsft0m__fQvecFT0MIm,O2qvectorsft0m__fSumAmplFT0M
121,True,-0.341987,0.657673,8953.869141
122,True,1.547365,0.428462,193459.390625
123,True,-0.671115,-0.580509,49.842911
124,True,0.426441,0.890671,57286.371094
125,True,1.399353,0.268564,12835.114258
126,True,-1.470713,-0.572693,16.576822
127,True,-0.574801,-0.886148,818.430542
128,True,-0.564174,-0.051474,3953.272949
129,True,0.040469,-0.544981,2310.823975
130,True,0.353821,0.171130,8679.996094
